In [ ]:
num_particles = 100
run_time_days = 20
time_step_minutes = 20
out_put_step_hours = 6

#initial position
lon0 = -50
lon1 = -48
lat0 = 1
lat1 = -0.5

depth_min = 1 #todo: figure near-surface depths 
depth_max = 10

offset = 1
ref_date = "1997-01-01"
#reproducibility
rdm_seed = 999

#paths
pathUVW= '/work/bk1450/b383184/Amazon/Mercator/data/variables'
# pathW= '/work/bk1450/b383184/Amazon/Atlantic/data/W'
pathTS = "/work/bk1450/b383184/Amazon/Mercator/data/variables"
Hgr = "/work/bk1450/b383184/Amazon/Mercator/data/Hgr_cmesh.nc"
Zgr = "/work/bk1450/b383184/Amazon/Mercator/data/Zgr_cmesh.nc"

In [ ]:
import numpy as np

In [ ]:
out_path = f'../data/tracks_{rdm_seed}/' #path to store the particle zarr

start_time = np.datetime64(ref_date) + np.timedelta64(offset, "D")
start_time

## Particles from the Plume to the Atlantic

* Release particles from the plume every 5 days for 4 years (1993-1999)
* Release time 1993 to 2013
* Number of particles =  10_000
* Release depth = (0,10)
* Tracking salinity and temperature

In [ ]:
from parcels import ParticleSet
from parcels import JITParticle
from parcels import AdvectionRK4_3D
from parcels import AdvectionRK4
from parcels import Variable
from datetime import timedelta
import numpy as np
from parcels import FieldSet
from glob import glob

In [ ]:
np.random.seed(rdm_seed)

### Copernicus Data (C grid)

In [ ]:
# ufiles = sorted(glob(f"{pathUVW}/U_1997.nc"))
# vfiles = sorted(glob(f"{pathUVW}/V_1997.nc"))
# wfiles = sorted(glob(f"{pathUVW}/W_1997.nc"))
# Tfiles = sorted(glob(f"{pathTS}/T_1997.nc"))
# Sfiles = sorted(glob(f"{pathTS}/S_1997.nc"))

ufiles = f"{pathUVW}/U_1997-01.nc"
vfiles = f"{pathUVW}/V_1997-01.nc"
wfiles = f"{pathUVW}/W_1997-01.nc"
Tfiles = f"{pathTS}/T_1997-01.nc"
Sfiles = f"{pathTS}/S_1997-01.nc"

In [ ]:
# print(len(ufiles))
# print(len(vfiles))
# print(len(wfiles))
# print(len(Tfiles))
# print(len(Sfiles))
ufiles

In [ ]:
## define the fieldset
filenames = {
    "U": {
        "data":ufiles,
        "lon": Hgr,
        "lat": Hgr,
        "depth": Zgr,
    },
    
    "V": {
        "data": vfiles,
        "lon": Hgr,
        "lat": Hgr,
        "depth": Zgr,
    },
    
    "W": {
        "data": wfiles,
        "lon": Hgr,
        "lat": Hgr,
        "depth": Zgr,
    },
    
    "T": {
        "data":Tfiles,
        "lon": Hgr,
        "lat": Hgr,
        "depth": Zgr,
    },
    
    "S": {  
        "data": Sfiles,
        "lon": Hgr,
        "lat": Hgr,
        "depth": Zgr,
    }

}

variables = {
    "U": "vozocrtx",
    "V": "vomecrty",
    "W": "vovecrtz",
    "T": "votemper",
    "S": "vosaline"
}

interp_method = {
    "U":"cgrid_velocity",
    "V":"cgrid_velocity",
    "W":"cgrid_velocity",
    "T": "linear", 
    "S": "linear"
}

dimensions = {
    "U": {"lon": "glamf", "lat": "gphif", "depth": "hdepw", "time": "time_counter"},
    "V": {"lon": "glamf", "lat": "gphif", "depth": "hdepw", "time": "time_counter"},
    "W": {"lon": "glamf", "lat": "gphif", "depth": "hdepw", "time": "time_counter"},
    "T": {"lon": "glamf", "lat": "gphif", "depth": "hdept", "time": "time_counter"},
    "S": {"lon": "glamf", "lat": "gphif", "depth": "hdept", "time": "time_counter"},
}


## now the fieldset
fieldset = FieldSet.from_netcdf(
    filenames, 
    variables, 
    dimensions,
    interp_method = interp_method,
    # tracer_interp_method="cgrid_tracer",
    deferred_load=True,
    allow_time_extrapolation = False,
    gridindexingtype = "nemo",
)

In [ ]:
start_pos_along_line = np.random.uniform(0,1,size=num_particles)
start_lon = lon0 + start_pos_along_line * (lon1-lon0)
start_lat = lat0 + start_pos_along_line * (lat1-lat0)
start_depth = np.random.uniform(depth_min,depth_max,size=num_particles)
start_times = np.datetime64(start_time)

In [ ]:
def SampleTS(particle, fieldset, time):
    # z = particle.depth
    # Optional: clamp depth to tracer range if needed
    # zmin, zmax can be added as constants if you want; otherwise omit.
    particle.temp = fieldset.T[time, particle.depth, particle.lat, particle.lon]
    particle.sal  = fieldset.S[time, particle.depth, particle.lat, particle.lon]

In [ ]:
## particle age
def Age(particle, fieldset, time):
    particle.age += particle.dt / 3600

In [ ]:
class SampleParticle(JITParticle):
    """
    Add variables to the standard particle class.
    Particles will sample temperature and salinity of the particle.
    """

    temp = Variable("temp", dtype=np.float32,initial=np.nan)
    sal = Variable("sal", dtype=np.float32,initial=np.nan)
    age = Variable("age", dtype=np.float32,initial=0.0)

In [ ]:
# initiate pset
pset = ParticleSet(
    fieldset=fieldset,
    pclass = SampleParticle,
    lon = start_lon,
    lat = start_lat,
    depth=start_depth,
    time=start_times
) 


out_fn = f'Parcels_run_{rdm_seed}_{start_time}.zarr'

output_file = pset.ParticleFile(
    name=out_path+out_fn,
    outputdt=timedelta(hours=out_put_step_hours),
    chunks = (num_particles,int(run_time_days*24/out_put_step_hours/4))
)

In [ ]:
##check the error
def CheckError(particle, fieldset, time):
    if particle.state >= 50:  # This captures all Errors
        particle.delete()

In [ ]:
## Execute particles
pset.execute(
    [Age,SampleTS,AdvectionRK4_3D,CheckError],
    runtime=timedelta(days=run_time_days),
    dt=timedelta(minutes=time_step_minutes),
    output_file= output_file
)

### Plotting

In [ ]:
import xarray as xr

In [ ]:
out_path = f'../data/tracks_{rdm_seed}/'
# out_fn = 'Parcels_run_692' 

ds_traj = xr.open_zarr(out_path+out_fn)
# ds_traj = ds_traj.compute()
ds_traj = ds_traj.compute()
ds_traj

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

last_valid = ds_traj.lat.notnull().astype(int).diff('obs', label='lower') == -1
ends = ds_traj.where(last_valid).mean('obs').compute()

fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.coastlines(resolution='110m')
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.gridlines(draw_labels=True, linestyle='--', linewidth=0.3)

ends.plot.scatter(
    x='lon',
    y='lat',
    hue='z' if 'z' in ends else None,
    cmap='viridis',
    s=30,
    ax=ax,
    transform=ccrs.PlateCarree()
)

ax.plot([lon0, lon1], [lat0, lat1])

plt.title('Mean last valid positions')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
ds_traj.lat.isnull().sum('trajectory').rename('Num_invalid').plot()
plt.show()

In [ ]:
fig, ax = plt.subplots()
sc = ax.scatter(ds_traj.lon.values, ds_traj.lat.values,
                c=ds_traj.sal.values, s=1, cmap='jet')
cb = fig.colorbar(sc, ax=ax, label='Sal')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.show()

In [ ]:
fig, ax = plt.subplots()
sc = ax.scatter(ds_traj.lon.values, ds_traj.lat.values,
                c=ds_traj.temp.values, s=1, cmap='jet')
cb = fig.colorbar(sc, ax=ax, label='temp')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.show()

In [ ]:
ds_traj.temp.min()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # registers 3D

lon  = np.asarray(ds_traj.lon.values).ravel()
lat  = np.asarray(ds_traj.lat.values).ravel()
z    = np.asarray(ds_traj.z.values).ravel()
temp = np.asarray(ds_traj.temp.values).ravel()

# Common finite mask (avoid length mismatch)
m = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(z) & np.isfinite(temp)

fig = plt.figure(figsize=(7,6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(lon[m], lat[m], z[m],
                c=temp[m], s=10, cmap='jet', edgecolors='none')

cb = fig.colorbar(sc, ax=ax, pad=0.1)
cb.set_label('Temperature')

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_zlabel('Depth (m)')
ax.invert_zaxis()
plt.tight_layout(); plt.show()

In [ ]:
cold = ds_traj.temp.values < 10  # °C
print("n cold points:", cold.sum())
print("example lon/lat/z:", ds_traj.lon.values[cold][:5],
      ds_traj.lat.values[cold][:5], ds_traj.z.values[cold][:5])

In [ ]:
fig = plt.figure(figsize=(7,6))
ax = fig.add_subplot(111, projection='3d')

sc = ax.scatter(ds_traj.lon.values[cold],ds_traj.lat.values[cold],-ds_traj.z.values[cold],c=ds_traj.temp.values[cold], 
                 marker='.',cmap='hot_r')
plt.colorbar(sc)
plt.show()

In [ ]:
fig = plt.figure(figsize=(7,6))
ax = fig.add_subplot(111, projection='3d')

sc = ax.scatter(ds_traj.lon.values[cold],ds_traj.lat.values[cold],-ds_traj.z.values[cold],c=ds_traj.age.values[cold]/24, 
                 marker='.',cmap='jet_r')
plt.colorbar(sc)
plt.show()
plt.title("Age [day]")

In [ ]:
import pandas as pd

df = pd.DataFrame({"lon":ds_traj.lon.values[cold],
                   "lat":ds_traj.lat.values[cold],
                   "z":ds_traj.z.values[cold]
                  })
df.to_csv(f'Cold_points_track_{rdm_seed}.csv')

In [ ]:
fieldset.V.grid

In [ ]:
ds_traj.z.plot()
plt.show()